<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-I/blob/main/Extra/datos_faltantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manipulacion de faltantes

In [ ]:
import pandas as pd
import numpy as np
url='https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv'
df= pd.read_csv(url,sep=',', header=None)
df.head()

**Variables incorporadas**

    0. Numero de veces embarazada (NEMB).
    1. Concentracion de plasma de glucosa (GLU)
    2. Presion arterial diastolica en mm Hg (PART).
    3. Grosor de piel en triceps en mm (GROS).
    4. 2-Hour serum insulin en mu U/ml (HUR).
    5. BMI (peso kg/(altura en m)^2 en (BMI).
    6. Funcion de prediccion de Diabetes (FPRED)
    7. Edad (años)  (AGE).
    8. Variable de clase (0 or 1)  (CLASS).


In [ ]:
# renombramos las columnas
df= df.rename(columns={0: "NEMB", 1: "GLU", 2: "PART",3:"GROS",4:"HUR",5:"BMI",6:"FPRED",7:"AGE",8:"CLASS"})
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
resumen=df.describe().round(1)
resumen

Se observan muchas columnas con valores minimos de 0 lo cual tiene poco sentido practico como por ejemplo:

    1. Concentracion de plasma de glucosa (GLU)
    2. Presion arterial diastolica en mm Hg (PART).
    3. Grosor de piel en triceps en mm (GROS).
    4. 2-Hour serum insulin en mu U/ml (HUR).
    5. BMI (peso kg/(altura en m)^2 en (BMI).


Calculemos cuantos valores problematicos tenemos

In [ ]:
# contar el numero de valores que son 0 por columnas
nun_missing = (df[['GLU','PART','GROS','HUR','BMI']] == 0).sum()
print(nun_missing)

Reemplazar estos valores por NaN

In [ ]:
# replace '0' values with 'nan'
df[['GLU','PART','GROS','HUR','BMI']] = df[['GLU','PART','GROS','HUR','BMI']].replace(0, np.nan)
# count the number of nan values in each column
print(df.isnull().sum())

In [ ]:
df[df.GLU.isna()]

In [ ]:
"""
La única diferencia entre isna() e isnull() es su nombre. isnull() es el nombre original de la función que se ha
utilizado en versiones anteriores de Pandas, mientras que isna() es un alias que se ha agregado luego
para mejorar la legibilidad del código.
"""

print(df.isna().sum())

In [ ]:
#que pasa si necesitamos reemplazar valores que son fisicamente sospechosos
url='https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv'
df2= pd.read_csv(url,sep=',', header=None)
df2= df2.rename(columns={0: "NEMB", 1: "GLU", 2: "PART",3:"GROS",4:"HUR",5:"BMI",6:"FPRED",7:"AGE",8:"CLASS"})
nun_missing_2 = (df2['GLU'] < 45).sum()
print(nun_missing_2)

In [ ]:
# reemplazar los valores menores que 45 por np.nan
df2['GLU'] = df2['GLU'].where(df2['GLU'] > 45, np.nan)
nun_missing_3 = (df2['GLU'] < 45).sum()
print(nun_missing_3)

In [ ]:
# otra forma de hacer lo mismo usando .loc
df2.loc[df2['GLU'] < 45, 'GLU'] = np.nan
nun_missing_3 = (df2['GLU'] < 45).sum()
print(nun_missing_3)

In [ ]:
df2

## Opcion 1 : Remover filas con valores faltantes

In [ ]:
# remover filas con na
df.dropna(inplace=True)
# mirar el resultado final
df.shape

In [ ]:
df

In [ ]:
# Reindexar el DataFrame con el índice predeterminado
df_reindexado = df.reset_index(drop=True)

# Mostrar el DataFrame reindexado
df_reindexado

$$\color{red}{\text{OJO!! Remover las filas solo porque tengan nulos no es la mejor forma de trabajar}}$$

## Opcion 2: Imputar los faltantes

Para esto tenemos varias opciones
1. Introducir un valor constante para los nulos o una categoria llamada Desconocido en variables categoricas
2. Reemplazar por un valor seleccionado al azar de los otros registros
3. Usar la media, mediana o moda para rellenar el valor
4. Valor estimado usando un modelo de ML

In [ ]:
url='https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv'
df= pd.read_csv(url,sep=',', header=None)
df= df.rename(columns={0: "NEMB", 1: "GLU", 2: "PART",3:"GROS",4:"HUR",5:"BMI",6:"FPRED",7:"AGE",8:"CLASS"})
print(df.shape)
df=df.replace(0, np.nan)
print(df.shape)

In [ ]:
print(df.head())

In [ ]:
# reemplazar con la mediana una columna
df.GLU.fillna("median", inplace=True)
print(df.shape)
print(df.head().round(2))

In [ ]:
df.GROS.fillna(-999, inplace=True)
print(df.shape)
print(df.head().round(2))

In [ ]:
df.GLU.isnull().sum()

In [ ]:
# reemplazar con la media
df.NEMB.fillna(df.NEMB.mean(), inplace=True)
print(df.shape)
print(df.head().round(2))

La librería ```scikit-learn``` proporciona la clase de preprocesamiento ```SimpleImputer``` que se puede usar para reemplazar los valores perdidos.

Es una clase flexible que le permite especificar el valor a reemplazar (puede ser algo diferente a NaN) y la técnica utilizada para reemplazarlo (como media, mediana o moda). La clase ```SimpleImputer``` opera directamente en la matriz NumPy en lugar del DataFrame.

El siguiente ejemplo usa la clase SimpleImputer para reemplazar los valores faltantes con la media de cada columna y luego imprime el número de valores de NaN en la matriz transformada.

In [ ]:
from numpy import nan
from numpy import isnan
from pandas import read_csv
from sklearn.impute import SimpleImputer
url='https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv'
df= pd.read_csv(url,sep=',', header=None)
df=df.replace(0, np.nan)
print(df.shape)
print(df.head())

In [ ]:
# Crear un numpy array con los valores

# definir el imputador
imputador = SimpleImputer(missing_values=np.nan, strategy='mean')
# transformar el dataset
transformados = imputador.fit_transform(df)



In [ ]:
transformados=pd.DataFrame(transformados)
#transformados= transformados.rename(columns={0: "NEMB", 1: "GLU", 2: "PART",3:"GROS",4:"HUR",5:"BMI",6:"FPRED",7:"AGE",8:"CLASS"})
print(transformados.head().round(2))
# Contar el numero de NaN en cada columna
print('Missing: ', transformados.isna().sum())

In [ ]:
imputador = SimpleImputer(missing_values=-999, strategy='mean')
# transformar el dataset
transformados = imputador.fit_transform(transformados)
# Contar el numero de NaN en cada columna
print('Missing: ', isnan(transformados).sum())

In [ ]:
transformados=pd.DataFrame(transformados)
#transformados= transformados.rename(columns={0: "NEMB", 1: "GLU", 2: "PART",3:"GROS",4:"HUR",5:"BMI",6:"FPRED",7:"AGE",8:"CLASS"})
print(transformados.head().round(2))

In [ ]:
# Crear un numpy array con los valores
valores = df.values
# definir el imputador
imputador = SimpleImputer(missing_values=np.nan, strategy='median')
# trabsformar el dataset
transformados = imputador.fit_transform(valores)
# Contar el numero de NaN en cada columna
print('Missing: %d' % isnan(transformados).sum())
transformados=pd.DataFrame(transformados)
#transformados= transformados.rename(columns={0: "NEMB", 1: "GLU", 2: "PART",3:"GROS",4:"HUR",5:"BMI",6:"FPRED",7:"AGE",8:"CLASS"})
print(transformados.head().round(2))

In [ ]:
# Crear un numpy array con los valores
valores = df.values
# definir el imputador
imputador = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
# trabsformar el dataset
transformados = imputador.fit_transform(valores)
# Contar el numero de NaN en cada columna
print('Missing: %d' % isnan(transformados).sum())
transformados=pd.DataFrame(transformados)
#transformados= transformados.rename(columns={0: "NEMB", 1: "GLU", 2: "PART",3:"GROS",4:"HUR",5:"BMI",6:"FPRED",7:"AGE",8:"CLASS"})
print(transformados.head().round(2))

In [ ]:
# Crear un numpy array con los valores
valores = df.values
# definir el imputador
imputador = SimpleImputer(missing_values=np.nan, strategy='constant',fill_value=10)
# trabsformar el dataset
transformados = imputador.fit_transform(valores)
# Contar el numero de NaN en cada columna
print('Missing: %d' % isnan(transformados).sum())
transformados=pd.DataFrame(transformados)
#transformados= transformados.rename(columns={0: "NEMB", 1: "GLU", 2: "PART",3:"GROS",4:"HUR",5:"BMI",6:"FPRED",7:"AGE",8:"CLASS"})
print(transformados.head().round(2))

In [ ]:
# Para obtener ayuda sobre la clase
SimpleImputer?

El parametro ```strategy``` puede tomar valores de ```['mean','median','most_frequent','constant']``` si se usa ``constant``` se debe proporcionar el argumento ```fill_value````con el valor a reemplazar